# FLUX

In [37]:
from dataclasses import dataclass

import torch
from torch import Tensor, nn
import math
from dataclasses import dataclass

from einops import rearrange

In [38]:
@dataclass
class FluxParams:
    in_channels: int  # número de canales por patch en la imagen de entrada.
    out_channels: int  # número de canales por patch en la imagen de salida.
    vec_in_dim: int  # dimensión del vector condicional.
    context_in_dim: int  # dimensión de los vectores de entrada textual.
    hidden_size: int  # dimensión interna (embedding) del modelo.
    mlp_ratio: float  # razón de expansión en los MLPs internos.
    num_heads: int  # número de cabezas de atención.
    depth: int  # cantidad de bloques duales (texto e imagen).
    depth_single_blocks: int  # cantidad de bloques fusionados posteriores.
    axes_dim: list[int]  # lista de dimensiones para codificación RoPE por eje.
    theta: int  # parámetro base de frecuencia RoPE.
    qkv_bias: bool  # indica si usar bias en las capas QKV.
    guidance_embed: bool  # indica si usar `guidance` como entrada adicional.

## Clase principal

In [ ]:
class Flux(nn.Module):

    def __init__(self, params: FluxParams):
        super().__init__()

        self.params = params
        self.in_channels = params.in_channels
        self.out_channels = params.out_channels
        self.hidden_size = params.hidden_size
        self.num_heads = params.num_heads

        assert params.hidden_size % params.num_heads == 0
        pe_dim = params.hidden_size // params.num_heads
        assert sum(params.axes_dim) == pe_dim

        # Proyecciones iniciales a una dimensión común:
        self.img_in = nn.Linear(self.in_channels, self.hidden_size)
        self.txt_in = nn.Linear(params.context_in_dim, self.hidden_size)
        self.time_in = MLPEmbedder(in_dim=256, hidden_dim=self.hidden_size)
        self.vector_in = MLPEmbedder(params.vec_in_dim, self.hidden_size)
        self.guidance_in = MLPEmbedder(in_dim=256, hidden_dim=self.hidden_size) if params.guidance_embed else nn.Identity()

        # Positional encoding:
        self.pe_embedder = EmbedND(dim=pe_dim, theta=params.theta, axes_dim=params.axes_dim)

        # Bloques principales:
        self.double_blocks = nn.ModuleList([DoubleStreamBlock(self.hidden_size, self.num_heads, params.mlp_ratio, params.qkv_bias) for _ in range(params.depth)])
        self.single_blocks = nn.ModuleList([SingleStreamBlock(self.hidden_size, self.num_heads, params.mlp_ratio) for _ in range(params.depth_single_blocks)])
        
        # Capa final:
        self.final_layer = LastLayer(self.hidden_size, 1, self.out_channels)


    def forward(self, img, img_ids, txt, txt_ids, timesteps, y, guidance=None):
        '''
        Entradas:
            img:       (B, L_img, in_channels)    — patches de imagen
            img_ids:   (B, L_img, A)              — coordenadas (ENTERAS) de cada patch de imagen
            txt:       (B, L_txt, context_in_dim) — embeddings de texto condicional
            txt_ids:   (B, L_txt, A)              — coordenadas posicionales ficticias para texto
            timesteps: (B,)                       — timestep de difusión
            y:         (B, vec_in_dim)            — vector de condición
            guidance:  (B,)                       — (opcional) intensidad del guidance

        Salida:
            tensor (B, L_img, out_channels)
        '''

        # Proyección de las entradas:
        img = self.img_in(img)                                        # (B, L_img, in_channels) -> (B, L_img, hidden_size)
        txt = self.txt_in(txt)                                        # (B, L_txt, context_in_dim) -> (B, L_txt, hidden_size)
        vec = self.vector_in(y)                                       # (B, vec_in_dim) -> (B, hidden_size)
        vec = vec + self.time_in(timestep_embedding(timesteps, 256))  # (B,) -> (B, 256) -> (B, hidden_size).
        
        # Inyectar guidance scale (mismo proceso que para inyectar timesteps):
        if self.params.guidance_embed:
            assert guidance is not None
            vec = vec + self.guidance_in(timestep_embedding(guidance, 256))  # (B,) -> (B, 256) -> (B, hidden_size)
        
        # Positional encoding:
        ids = torch.cat((txt_ids, img_ids), dim=1)  # (B, L_txt + L_img, A)
        pe = self.pe_embedder(ids)  # (B, L_txt + L_img, head_dim, 2)

        # Bloques de atención dual (con modulación):
        for block in self.double_blocks: 
            img, txt = block(img=img, txt=txt, vec=vec, pe=pe)  # entrada y salida: (B, L_img, hidden_size), (B, L_txt, hidden_size).

        # Fusión y bloques de atención simple (con modulación):
        img = torch.cat((txt, img), dim=1)  # (B, L_txt + L_img, hidden_size)
        for block in self.single_blocks:
            img = block(img, vec=vec, pe=pe)  # entrada y salida: (B, L_txt + L_img, hidden_size)
        img = img[:, txt.shape[1]:, ...]  # (B, L_img, hidden_size)

        # Proyección final (con normalización adaptativa):
        img = self.final_layer(img, vec)  # (B, L_img, hidden_size) -> (B, L_img, patch_size * patch_size * out_channels), con patch_size * patch_size * out_channels la cantidad de píxeles en cada patch.
        return img

## Módulos secundarios

### `MLPEmbedder`

MLP de dos capas `in_dim` -> `hidden_dim` -> `hidden_dim`. Usada para proyectar los embedding del timestep, del vector condicional y del guidance.

In [40]:
class MLPEmbedder(nn.Module):

    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.in_layer = nn.Linear(in_dim, hidden_dim)
        self.silu = nn.SiLU()
        self.out_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        """
        Entrada:
            x (B, in_dim)

        Salida:
            tensor (B, hidden_dim)
        """
        return self.out_layer(self.silu(self.in_layer(x)))

### `EmbedND`

- Calcula codificaciones posicionales RoPE en múltiples dimensiones (ND).

In [ ]:
class EmbedND(nn.Module):

    def __init__(self, dim: int, theta: int, axes_dim: list[int]):
        super().__init__()
        self.dim = dim
        self.theta = theta
        self.axes_dim = axes_dim

    def forward(self, ids):
        """
        Entrada:
            ids  (B, L, A) — índices posicionales, donde A es el número de ejes.

        Salida:
            Tensor (B, 1, L, D, 2) — codificación ROPE concatenada en la dimensión D,
                con D = sum(axes_dim), y la última dimensión corresponde a (cos, sin)
        """
        n_axes = ids.shape[-1]
        emb = torch.cat([rope(ids[..., i], self.axes_dim[i], self.theta) for i in range(n_axes)], dim=-3)
        return emb.unsqueeze(1)

### `DoubleStreamBlock`

- Atención dual: bloque de atención dual (imagen y texto) con modulación condicional y MLP.

In [ ]:
class DoubleStreamBlock(nn.Module):

    def __init__(self, hidden_size, num_heads, mlp_ratio, qkv_bias):
        super().__init__()

        mlp_hidden_dim = int(hidden_size * mlp_ratio)
        self.num_heads = num_heads
        self.hidden_size = hidden_size

        # Imagen:
        self.img_norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.img_mod = Modulation(hidden_size, double=True)
        self.img_attn = SelfAttention(dim=hidden_size, num_heads=num_heads, qkv_bias=qkv_bias)
        self.img_norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.img_mlp = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(mlp_hidden_dim, hidden_size),
        )

        # Texto (exactamente igual que imagen):
        self.txt_norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.txt_mod = Modulation(hidden_size, double=True)
        self.txt_attn = SelfAttention(dim=hidden_size, num_heads=num_heads, qkv_bias=qkv_bias)
        self.txt_norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.txt_mlp = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden_dim, bias=True),
            nn.GELU(approximate="tanh"),
            nn.Linear(mlp_hidden_dim, hidden_size, bias=True),
        )

    def forward(self, img, txt, vec, pe):
        """

        Entradas:
            img (B, L_img, hidden_size)
            txt (B, L_txt, hidden_size)
            vec (B, hidden_size)
            pe: (B, L_img + L_txt, head_dim, 2)

        Salidas:
            img (B, L_img, hidden_size)
            txt (B, L_txt, hidden_size)
        """

        # Imagen (normalización -> modulación -> calcular QKV):
        img_mod1, img_mod2 = self.img_mod(vec)
        img_modulated = self.img_norm1(img)
        img_modulated = (1 + img_mod1.scale) * img_modulated + img_mod1.shift  # (B, L_img, hidden_size)
        img_qkv = self.img_attn.qkv(img_modulated)  # (B, L_img, 3 * hidden_size)
        img_q, img_k, img_v = rearrange(img_qkv, "B L (K H D) -> K B H L D", K=3, H=self.num_heads)  # cada uno (B, num_heads, L_img, head_dim).
        img_q, img_k = self.img_attn.norm(img_q, img_k, img_v)  # cada uno (B, num_heads, L_img, head_dim).

        # Texto (exactamente igual que imagen):
        txt_mod1, txt_mod2 = self.txt_mod(vec)
        txt_modulated = self.txt_norm1(txt)
        txt_modulated = (1 + txt_mod1.scale) * txt_modulated + txt_mod1.shift  # (B, L_txt, hidden_size)
        txt_qkv = self.txt_attn.qkv(txt_modulated)  # (B, L_txt, 3 * hidden_size)
        txt_q, txt_k, txt_v = rearrange(txt_qkv, "B L (K H D) -> K B H L D", K=3, H=self.num_heads)  # cada uno (B, num_heads, L_txt, head_dim).
        txt_q, txt_k = self.txt_attn.norm(txt_q, txt_k, txt_v)  # cada uno (B, num_heads, L_txt, head_dim).

        # Atención cruzada (texto e imagen):
        q = torch.cat((txt_q, img_q), dim=2)  # (B, num_heads, L_txt + L_img, head_dim)
        k = torch.cat((txt_k, img_k), dim=2)  # (B, num_heads, L_txt + L_img, head_dim)
        v = torch.cat((txt_v, img_v), dim=2)  # (B, num_heads, L_txt + L_img, head_dim)
        attn = rope_attention(q, k, v, pe=pe)  # (B, L_txt + L_img, hidden_size)
        print(attn.shape)
        txt_attn, img_attn = attn[:, : txt.shape[1]], attn[:, txt.shape[1] :]

        # Imagen (proyección attn + modulación + conexión residual; norm + modulación + MLP final):
        img = img + img_mod1.gate * self.img_attn.proj(img_attn)
        img = img + img_mod2.gate * self.img_mlp((1 + img_mod2.scale) * self.img_norm2(img) + img_mod2.shift)

        # Texto (exactamente igual que imagen):
        txt = txt + txt_mod1.gate * self.txt_attn.proj(txt_attn)
        txt = txt + txt_mod2.gate * self.txt_mlp((1 + txt_mod2.scale) * self.txt_norm2(txt) + txt_mod2.shift)

        return img, txt

### `SingleStreamBlock`

Fused attention: bloque transformer tipo DiT con atención + MLP en paralelo y modulación adaptativa, siguiendo la arquitectura de ViT 22B.


In [ ]:
class SingleStreamBlock(nn.Module):

    def __init__(self,hidden_size, num_heads, mlp_ratio, qk_scale=None):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        head_dim = hidden_size // num_heads
        self.scale = qk_scale or head_dim**-0.5
        self.mlp_hidden_dim = int(hidden_size * mlp_ratio)

        self.modulation = Modulation(hidden_size, double=False)
        self.pre_norm = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)

        self.linear1 = nn.Linear(hidden_size, hidden_size * 3 + self.mlp_hidden_dim)  # capa lineal para QKV y MLP en paralelo.
        self.linear2 = nn.Linear(hidden_size + self.mlp_hidden_dim, hidden_size)  # proyección y MLP salida.

        self.norm = QKNorm(head_dim)
        
        self.mlp_act = nn.GELU(approximate="tanh")
        

    def forward(self, x: Tensor, vec: Tensor, pe: Tensor) -> Tensor:
        """
        Entradas:
            x    (B, L, hidden_size) — secuencia fusionada (imagen + texto).
            vec  (B, hidden_size)
            pe   (B, L, head_dim, 2) — codificación ROPE con head_dim = hidden_size // num_heads

        Salida:
            tensor (B, L, hidden_size)
        """

        mod, _ = self.modulation(vec)  # atributos shift, scale, gate. Cada uno de dimensión (B, 1, hidden_size).
        x_mod = (1 + mod.scale) * self.pre_norm(x) + mod.shift  # (B, L, hidden_size).

        # Proyección QKV y MLP:
        qkv, mlp = torch.split(self.linear1(x_mod), [3 * self.hidden_size, self.mlp_hidden_dim], dim=-1)  # (B, L, 3*hidden_size), (B, L, mlp_hidden_dim).
        q, k, v = rearrange(qkv, "B L (K H D) -> K B H L D", K=3, H=self.num_heads)  # cada uno de tamaño (B, num_heads, L, head_dim)
        
        # Atención con QKNorm y RoPE:
        q, k = self.norm(q, k, v)  # (B, num_heads, L, head_dim), (B, num_heads, L, head_dim).
        attn = rope_attention(q, k, v, pe=pe)  # (B, L, hidden_size).

        # FF paralelo + modulación final:
        out = self.linear2(torch.cat((attn, self.mlp_act(mlp)), dim=2))  # (B, L, hidden_size + mlp_hidden_dim) -> (B, L, hidden_size).
        out = x + mod.gate * out  # (B, L, hidden_size).

        return out

### `LastLayer`

In [44]:
class LastLayer(nn.Module):

    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 2 * hidden_size)
        )
        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels)

    def forward(self, img, vec):
        """
        Entradas:
            img (B, L_img, hidden_size)
            vec (B, hidden_size)

        Salida:
            tensor (B, L_img, patch_size * patch_size * out_channels)
        """

        shift, scale = self.adaLN_modulation(vec).chunk(2, dim=1)  # (B, hidden_size) -> (B, hidden_size), (B, hidden_size).
        img = (1 + scale[:, None, :]) * self.norm_final(img) + shift[:, None, :]  # (B, L_img, hidden_size).
        img = self.linear(img)  # (B, L_img, hidden_size) -> (B, L_img, patch_size * patch_size * out_channels)
        return img

## Módulos terciarios

### `QKNorm`

- Las normalizaciones como RMSNorm se suelen computar en float32, aunque el modelo completo puede estar en float16 o bfloat16 (por ejemplo en inferencia con torch.autocast). Al hacer .to(v) te aseguras de volver al tipo correcto antes de seguir. Equivale a to(dtype=v.dtype, device=v.device).

In [45]:
class QKNorm(torch.nn.Module):
    """
    Normalización independiente para queries y keys (RMSNorm compartido por batch).

    Entradas:
        q (B, num_heads, L, head_dim)
        k (B, num_heads, L, head_dim)
        v (B, num_heads, L, head_dim) — solo para tipado y compatibilidad

    Salida:
        q_norm (B, num_heads, L, head_dim)
        k_norm (B, num_heads, L, head_dim)
    """
    def __init__(self, dim: int):
        super().__init__()
        self.query_norm = RMSNorm(dim)
        self.key_norm = RMSNorm(dim)

    def forward(self, q, k, v):
        q = self.query_norm(q)
        k = self.key_norm(k)
        return q.to(v), k.to(v)

### `RMSNorm`

- RMSNorm solo divide por la norma y agrega un parámetro de escala.
- Se aplica independentemente a cada vector de características (i.e., en cada posición de la secuencia, en cada cabezal y en cada elemento del batch).

In [46]:
class RMSNorm(torch.nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        """
        Entrada:
            x (B, *, dim)

        Salida:
            tensor (B, *, dim)
        """
        x_dtype = x.dtype
        x = x.float()
        rrms = torch.rsqrt(torch.mean(x**2, dim=-1, keepdim=True) + 1e-6)  # (B, *, 1)
        return (x * rrms).to(dtype=x_dtype) * self.scale

### `Modulation`

In [47]:
@dataclass
class ModulationOut:
    shift: Tensor
    scale: Tensor
    gate: Tensor

class Modulation(nn.Module):

    def __init__(self, dim, double: bool):
        super().__init__()
        self.is_double = double
        self.multiplier = 6 if double else 3  # (shift, scale, gate) * 2 si double==True.
        self.lin = nn.Linear(dim, self.multiplier * dim)

    def forward(self, vec):
        """
        Entrada:
            vec  (B, dim) — vector de contexto embebido (por ejemplo, tiempo + guidance + y)

        Salidas:
            1 (double==False) o 2 (double==True) objetos ModulationOut. Cada uno con shift, scale, gate de tamaño (B, 1, dim).
        """
        # out: (B, dim) -> (B, multiplier * dim) -> (B, 1, multiplier * dim) -> (B, 1, dim) * multiplier.
        out = self.lin(nn.functional.silu(vec))[:, None, :].chunk(self.multiplier, dim=-1)  
        mod1 = ModulationOut(*out[:3])
        mod2 = ModulationOut(*out[3:]) if self.is_double else None
        return (mod1, mod2)

### `SelfAttention`

In [48]:
class SelfAttention(nn.Module):

    def __init__(self, dim: int, num_heads: int = 8, qkv_bias: bool = False):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.norm = QKNorm(head_dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x: Tensor, pe: Tensor) -> Tensor:
        """
        Módulo de MHSA con RoPE y QKNorm.

        Entradas (L: largo secuencia, D: dimensión características):
            x: Tensor de forma (B, L, D) — entrada embebida (por ejemplo, imagen o texto)
            pe: Tensor de forma (B, L, D_head, 2) — codificación posicional ROPE, donde D_head = D // num_heads

        Salida:
            Tensor de forma (B, L, D) — resultado de atención proyectado.
        """
        qkv = self.qkv(x)  # (B, L, 3 * D)
        q, k, v = rearrange(qkv, "B L (K H D) -> K B H L D", K=3, H=self.num_heads)  # K: n_chunks, D: head_dim; cada elemento es (B, H, L, head_dim).
        q, k = self.norm(q, k, v)
        x = rope_attention(q, k, v, pe=pe)  # (B, L, H*head_dim=dim).
        x = self.proj(x)  # (B, L, D)
        return x

## Funciones auxiliares

### `attention`

Cálculo de atención con RoPE

- Aplica atención con rope a los tensores q, k, v.
- El mecanismo de atención se aplica independientemente sobre cada elemento del batch y cada cabezal de atención.
- A la salida concatena las H cabezas en la dimensión de características (cada token tiene H*D características).

In [ ]:
def rope_attention(q, k, v, pe):  # se usa en SingleStreamBlock y Double.
    """
    Entradas:
        q (B, num_heads, L, head_dim)
        k (B, num_heads, L, head_dim)
        v (B, num_heads, L, head_dim)
        pe (B, 1, L, 1, 2, 2)
    Salida:
        Tensor de forma (B, L, hidden_size)
    """

    q, k = apply_rope(q, k, pe)  # ambos (B, num_heads, L, head_dim).
    x = torch.nn.functional.scaled_dot_product_attention(q, k, v)  # (B, num_heads, L, head_dim)
    x = rearrange(x, "B H L D -> B L (H D)")  # (B, L, num_heads * head_dim=hidden_size)
    return x

### `apply_rope`

- El pe se aplica independientemente a cada cabezal.
- `apply_rope` aplica RoPE a cada cabezal 

In [ ]:
def apply_rope(xq: Tensor, xk: Tensor, freqs_cis: Tensor) -> tuple[Tensor, Tensor]:
    """
    Aplica rotación posicional (ROPE) a tensores q y k.

    Entradas:
        xq: Tensor de forma (B, H, L, D) — queries
        xk: Tensor de forma (B, H, L, D) — keys
        freqs_cis: Tensor de forma (B, L, D, 2) — parámetros ROPE para cos/sin

    Salidas:
        xq_out: Tensor de forma (B, H, L, D)
        xk_out: Tensor de forma (B, H, L, D)
    """
    xq_ = xq.float().reshape(*xq.shape[:-1], -1, 1, 2)
    xk_ = xk.float().reshape(*xk.shape[:-1], -1, 1, 2)
    xq_out = freqs_cis[..., 0] * xq_[..., 0] + freqs_cis[..., 1] * xq_[..., 1]
    xk_out = freqs_cis[..., 0] * xk_[..., 0] + freqs_cis[..., 1] * xk_[..., 1]
    return xq_out.reshape(*xq.shape).type_as(xq), xk_out.reshape(*xk.shape).type_as(xk)

### `rope`

In [ ]:
def rope(pos: Tensor, dim: int, theta: int) -> Tensor:
    """
    Calcula codificación rotacional posicional (ROPE) para un eje.
    Genera matrices de rotación seno-coseno por dimensión.

    Entradas:
        pos: Tensor de forma (B, L) o (B, L, n) — posición discreta o continua
        dim: Entero, dimensión de salida por eje (debe ser par)
        theta: Parámetro de frecuencia base

    Salida:
        Tensor de forma (B, L, dim//2, 2, 2) — matriz de rotación para ROPE
    """
    assert dim % 2 == 0
    scale = torch.arange(0, dim, 2, dtype=pos.dtype, device=pos.device) / dim
    omega = 1.0 / (theta**scale)
    out = torch.einsum("...n,d->...nd", pos, omega)
    out = torch.stack([torch.cos(out), -torch.sin(out), torch.sin(out), torch.cos(out)], dim=-1)
    out = rearrange(out, "b n d (i j) -> b n d i j", i=2, j=2)
    return out.float()

### `timestep_embedding`

In [52]:
def timestep_embedding(t: Tensor, dim: int, max_period=10000, time_factor=1000.0) -> Tensor:
    """
    Genera embeddings sinusoidales continuos para timesteps.

    Entradas:
        t: (B,). timestep continuo o discreto
        dim: dimensión de salida.
        max_period: frecuencia mínima considerada
        time_factor: factor de escala del tiempo

    Salida:
        Tensor de forma (B, dim) — embedding posicional
    """
    t = time_factor * t
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half).to(t.device)
    args = t[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    if torch.is_floating_point(t):
        embedding = embedding.to(t)
    return embedding

## Ejemplo de `Flux`

In [53]:
params = FluxParams(
    in_channels=3,
    out_channels=3,
    vec_in_dim=10,
    context_in_dim=5,
    hidden_size=64,
    mlp_ratio=4.0,
    num_heads=4,
    depth=2,
    depth_single_blocks=1,
    axes_dim=[16],  # sum debe ser igual a hidden_size // num_heads = 16
    theta=10000,
    qkv_bias=True,
    guidance_embed=True,
)

# Parámetros de entrada:
batch_size = 8
seq_img = 7   # longitud de secuencia imagen
seq_txt = 5    # longitud de secuencia texto

# Tensores aleatorios:
img = torch.randn(batch_size, seq_img, params.in_channels)
img_ids = torch.randint(0, 100, (batch_size, seq_img, 1))
txt = torch.randn(batch_size, seq_txt, params.context_in_dim)
txt_ids = torch.randint(0, 100, (batch_size, seq_txt, 1))
timesteps = torch.rand(batch_size)
y = torch.randn(batch_size, params.vec_in_dim)
guidance = torch.rand(batch_size)

# Forward:
model = Flux(params)
output = model(img, img_ids, txt, txt_ids, timesteps, y, guidance)
assert output.shape == (batch_size, seq_img, params.out_channels)

torch.Size([8, 12, 64])
torch.Size([8, 12, 64])


In [54]:
class FluxLoraWrapper(Flux):
    def __init__(
        self,
        lora_rank: int = 128,
        lora_scale: float = 1.0,
        *args,
        **kwargs,
    ) -> None:
        super().__init__(*args, **kwargs)

        self.lora_rank = lora_rank

        replace_linear_with_lora(
            self,
            max_rank=lora_rank,
            scale=lora_scale,
        )

    def set_lora_scale(self, scale: float) -> None:
        for module in self.modules():
            if isinstance(module, LinearLora):
                module.set_scale(scale=scale)